# Gear 2.2 · paired state vs fill (frozen anchors)

Outputs: `research/output/gear22_random_anchor_persistence_paired/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('research/output/gear22_random_anchor_persistence_paired')
print(json.dumps(json.loads((OUT/'verdict.json').read_text()), indent=2))
s = pd.read_csv(OUT/'paired_summary.csv')
s100 = s[s.L_ms==100]

In [ ]:
def pool(df, zb):
    g = df[(df.z_bin==zb) & (df.n_paired>0)]
    rows=[]
    for (W,L), gg in g.groupby(['W_ms','L_ms']):
        w = gg.n_paired.to_numpy(float)
        rows.append({
            'W_ms':W,'L_ms':L,
            'A_state': np.average(gg.A_state, weights=w),
            'A_fill': np.average(gg.A_fill, weights=w),
            'A_cadence': np.average(gg.A_cadence, weights=w),
            'n': int(w.sum()),
        })
    return pd.DataFrame(rows)

fig, axes = plt.subplots(1,2, figsize=(10,4), sharey=True)
for ax, zb in zip(axes, ['z_2_4','z_gt_4']):
    P = pool(s, zb)
    for W, ls in [(0,'-'), (100,':')]:
        g = P[P.W_ms==W].sort_values('L_ms')
        ax.plot(g.L_ms, g.A_state, ls, label=f'A_state W={W}')
        ax.plot(g.L_ms, g.A_fill, ls, alpha=0.7, label=f'A_fill W={W}')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(zb); ax.set_xlabel('L ms'); ax.legend(fontsize=7)
fig.suptitle('Paired A_state vs A_fill')
plt.tight_layout(); plt.show()

In [ ]:
cols = ['side','W_ms','z_bin','n_paired','A_state','A_fill','A_cadence',
        'ci_A_state_low','ci_A_state_high','ci_A_fill_low','ci_A_fill_high']
s100[s100.z_bin.isin(['z_2_4','z_gt_4'])][cols].sort_values(['z_bin','side','W_ms'])